In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from preprocessing import dt_product_df, preprocess_text

C:\Users\jeffr\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ud-anthony.vpnstores.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\jeffr\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ud-anthony.vpnstores.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [2]:
### CONTENT BASED FILTERING ###
# Preprocessing kolom teks
text_colums_content = ['gender', 'skin_type_face', 'hair_issue', 'skin_type_body']

for column in text_colums_content:
    dt_product_df[column] = dt_product_df[column].apply(preprocess_text)

grouped_data = dt_product_df.groupby(['category_id', 'subcategory_id'])

In [3]:
# Fungsi untuk menghitung TF-IDF dan similaritas kosinus serta memberikan rekomendasi
def calculate_similarity(group, user_id):
    # Inisialisasi TF-IDF Vectorizer
    tfidf_vectorizer = TfidfVectorizer()

    # Ambil atribut untuk perhitungan (skin_type)
    attributes = group[['skin_type_face', 'hair_issue', 'skin_type_body']].astype(str).apply(lambda x: ' '.join(x), axis=1)

    # Hitung TF-IDF
    tfidf_matrix = tfidf_vectorizer.fit_transform(attributes)

    # Ambil data produk yang sudah dirating oleh user
    rated_products = set(group[group['user_id'] == user_id]['id'])

    # Jika user belum melakukan rating, kembalikan array kosong
    if not rated_products:
        return []
    
    # Ambil jumlah total rating dan rata-rata rating
    total_ratings = group['rating'].sum()
    average_rating = group['rating'].mean()

    # Inisialisasi list untuk menyimpan rekomendasi produk
    recommendations = []

    # Ambil indeks produk yang dirating oleh user
    rated_indices = [idx for idx, product_id in enumerate(group['id']) if product_id in rated_products]

    # Lakukan iterasi melalui setiap produk yang dirating oleh user
    for query_index in rated_indices:
        # Ambil query dan lakukan reshape
        query = tfidf_matrix[query_index]

        # Hitung similaritas kosinus antara query dan semua produk
        cosine_similarities = cosine_similarity(query, tfidf_matrix).flatten()

        # Urutkan indeks produk berdasarkan similaritas kosinus
        similar_indices = cosine_similarities.argsort()[::-1]

        # Tambahkan produk, termasuk yang sudah dirating oleh user, ke dalam list recommendations
        for idx in similar_indices:
            # Hitung bobot rekomendasi berdasarkan nilai similaritas, bobot TF-IDF, jumlah rating, dan rata-rata rating
            recommendation_weight = (0.3 * tfidf_matrix[idx, :].sum() + 
                                     0.6 * cosine_similarities[idx] + 
                                     0.05 * (group.iloc[idx]['rating'] / total_ratings) + 
                                     0.05 * (group.iloc[idx]['rating'] / average_rating))
            recommendations.append((group.iloc[idx]['id'], recommendation_weight))
            if len(recommendations) >= 16:
                break

        if len(recommendations) >= 16: 
            break

    # Urutkan rekomendasi berdasarkan nilai similaritas tertinggi
    recommendations.sort(key=lambda x: x[1], reverse=True)

    # Mengambil hanya id produk dari rekomendasi
    recommended_product_ids = [rec[0] for rec in recommendations]

    # Mengembalikan rekomendasi
    return recommended_product_ids

In [6]:
# Iterate through each user and category to get recommendations
user_ids = dt_product_df['user_id'].unique()
recommendations_by_user = {}

for user_id in user_ids:
    user_recommendations = {}
    for (category_id, subcategory_id), group in grouped_data:
        recommendations = calculate_similarity(group, user_id)
        user_recommendations[(category_id, subcategory_id)] = recommendations
    recommendations_by_user[user_id] = user_recommendations

# Display recommendations for each user
for user_id, categories in recommendations_by_user.items():
    print(f"User ID: {user_id}")
    for (category_id, subcategory_id), recommendations in categories.items():
        print(f"  Category ID: {category_id}, Subcategory ID: {subcategory_id}")
        print(f"  Recommended Products: {recommendations}")
    print()

User ID: 32.0
  Category ID: 1, Subcategory ID: 1
  Recommended Products: [51, 54, 56, 46, 46, 50, 49, 47, 58, 46, 57, 51, 47, 56, 57, 56]
  Category ID: 1, Subcategory ID: 2
  Recommended Products: [171, 173, 215, 211, 224, 218, 219, 255, 219, 199, 199, 199, 173, 209, 215, 211]
  Category ID: 1, Subcategory ID: 3
  Recommended Products: [61, 59, 60, 61, 61, 59, 61, 59, 59, 61, 61, 60, 61, 61, 59, 61]
  Category ID: 1, Subcategory ID: 4
  Recommended Products: []
  Category ID: 1, Subcategory ID: 5
  Recommended Products: [69, 78, 78, 72, 78, 78, 78, 71, 69, 70, 78, 78, 71, 78, 73, 78]
  Category ID: 1, Subcategory ID: 7
  Recommended Products: [79, 82, 82, 82, 81, 83, 84, 81, 81, 82, 82, 84, 82, 81, 82, 81]
  Category ID: 1, Subcategory ID: 8
  Recommended Products: [87, 87, 92, 95, 90, 92, 87, 91, 90, 89, 94, 95, 87, 87, 94, 88]
  Category ID: 1, Subcategory ID: 9
  Recommended Products: [62, 62, 63, 62, 62, 62, 62, 62, 63, 62, 62, 63, 63, 63, 63, 62]
  Category ID: 2, Subcategory ID